# Multimodal AI Assistant

GROUP MEMBERS;

Shaadi Rashid Ssemuddu 202301518

Awab Mohamedosman EisaMohamedosman 202302335

Abdullah Nasser MohammedAl-Amri 202201658

This demo uses one prompt to generate text with GPT-2, improve the best response using human feedback, and generate an image with Stable Diffusion.

**Models**

- Text: `openai-community/gpt2`
- Image: `stable-diffusion-v1-5/stable-diffusion-v1-5`

## Setup



In [ ]:
!pip -q install transformers diffusers accelerate safetensors torch pillow matplotlib pandas

In [ ]:
#python standard library modules
import json
import random
from datetime import datetime
from pathlib import Path
#External libraries
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, set_seed

SEED = 42
TEXT_MODEL_ID = "openai-community/gpt2"
DIFFUSION_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

OUTPUT_DIR = Path("project_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FEEDBACK_FILE = OUTPUT_DIR / "rlhf_feedback.csv"
IMPROVEMENT_FEEDBACK_FILE = OUTPUT_DIR / "rlhf_improvement_feedback.csv"
IMAGE_FILE = OUTPUT_DIR / "generated_diffusion_image.png"

random.seed(SEED)
set_seed(SEED)

def get_best_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = get_best_device()
print(f"Device: {DEVICE}")

## Prompt

The same prompt is used for text and image generation.

In [ ]:
USER_PROMPT = "A futuristic smart city with flying cars at sunset"
display(Markdown(f"> {USER_PROMPT}"))

# 1. Text Generation

In [ ]:
def load_text_generator(model_id=TEXT_MODEL_ID):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id

    device_index = 0 if DEVICE == "cuda" else -1
    return pipeline("text-generation", model=model, tokenizer=tokenizer, device=device_index), tokenizer

text_generator, text_tokenizer = load_text_generator()
print("GPT-2 loaded.")

In [ ]:
def clean_text(text):
    return " ".join(text.replace("\n", " ").split()).strip()

def generate_text_responses(prompt, count=3):
    generation_prompt = f"Write a vivid short paragraph describing this scene: {prompt}\nParagraph:"
    outputs = text_generator(
        generation_prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.9,
        top_p=0.92,
        top_k=50,
        num_return_sequences=count,
        return_full_text=False,
        pad_token_id=text_tokenizer.eos_token_id,
    )
    return [
        {"response_id": idx, "text": clean_text(output["generated_text"])}
        for idx, output in enumerate(outputs, start=1)
    ]

responses = generate_text_responses(USER_PROMPT)

for response in responses:
    display(Markdown(f"### Response {response['response_id']}\n{response['text']}"))

# 2. Human Feedback

In [ ]:
def ask_rating(label):
    while True:
        value = input(f"Rate {label} from 1 to 5: ").strip()
        try:
            rating = int(value)
            if 1 <= rating <= 5:
                return rating
        except ValueError:
            pass
        print("Enter a number from 1 to 5.")

def ask_best_response(valid_ids):
    valid_ids = set(valid_ids)
    while True:
        value = input(f"Which response is best? {sorted(valid_ids)}: ").strip()
        try:
            selected = int(value)
            if selected in valid_ids:
                return selected
        except ValueError:
            pass
        print("Enter one of the response numbers.")

def save_feedback(new_feedback, feedback_file):
    if feedback_file.exists():
        old_feedback = pd.read_csv(feedback_file)
        all_feedback = pd.concat([old_feedback, new_feedback], ignore_index=True)
    else:
        all_feedback = new_feedback.copy()
    all_feedback.to_csv(feedback_file, index=False)
    return all_feedback

def collect_feedback(prompt, generated_responses):
    ratings = {
        item["response_id"]: ask_rating(f"Response {item['response_id']}")
        for item in generated_responses
    }
    best_id = ask_best_response([item["response_id"] for item in generated_responses])
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    rows = []
    for item in generated_responses:
        response_id = item["response_id"]
        selected_best = int(response_id == best_id)
        rating = ratings[response_id]
        rows.append({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "run_id": run_id,
            "prompt": prompt,
            "response_id": response_id,
            "response_text": item["text"],
            "rating": rating,
            "selected_best": selected_best,
            "reward_score": rating + (0.5 * selected_best),
        })
    return pd.DataFrame(rows)

feedback_df = collect_feedback(USER_PROMPT, responses)
all_feedback_df = save_feedback(feedback_df, FEEDBACK_FILE)
feedback_df

# 3. Ranking

In [ ]:
def rank_responses(all_feedback, prompt):
    data = all_feedback[all_feedback["prompt"] == prompt].copy()
    ranking = (
        data.groupby(["prompt", "response_text"], as_index=False)
        .agg(
            average_rating=("rating", "mean"),
            best_votes=("selected_best", "sum"),
            average_reward=("reward_score", "mean"),
            feedback_count=("rating", "count"),
        )
        .sort_values(["average_reward", "average_rating", "best_votes"], ascending=False)
        .reset_index(drop=True)
    )
    ranking.insert(0, "rank", range(1, len(ranking) + 1))
    return ranking

ranking_df = rank_responses(all_feedback_df, USER_PROMPT)
ranking_df

# 4. Improve the Best Response

The top-ranked response is rewritten using the human preference signal.

In [ ]:
def improve_best_response(prompt, ranking):
    best_text = ranking.iloc[0]["response_text"]
    improvement_prompt = (
        "Improve this response so it is vivid, concise, coherent, and clearly related to the prompt.\n\n"
        f"Prompt: {prompt}\n"
        f"Original response: {best_text}\n"
        "Improved response:"
    )
    output = text_generator(
        improvement_prompt,
        max_new_tokens=90,
        do_sample=True,
        temperature=0.75,
        top_p=0.9,
        num_return_sequences=1,
        return_full_text=False,
        pad_token_id=text_tokenizer.eos_token_id,
    )[0]["generated_text"]
    return clean_text(output)

improved_response = improve_best_response(USER_PROMPT, ranking_df)

display(Markdown("### Original Winner"))
display(Markdown(ranking_df.iloc[0]["response_text"]))
display(Markdown("### Improved Response"))
display(Markdown(improved_response))

# 5. Second Feedback Round

In [ ]:
def ask_yes_no(question):
    while True:
        value = input(question).strip().lower()
        if value in {"yes", "y"}:
            return 1
        if value in {"no", "n"}:
            return 0
        print("Enter yes or no.")

def collect_improvement_feedback(prompt, ranking, improved_text):
    original_text = ranking.iloc[0]["response_text"]
    original_rating = float(ranking.iloc[0]["average_rating"])
    improved_rating = ask_rating("the improved response")
    preferred = ask_yes_no("Is the improved response better than the original winner? (yes/no): ")

    return pd.DataFrame([
        {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "prompt": prompt,
            "original_response": original_text,
            "original_average_rating": original_rating,
            "improved_response": improved_text,
            "improved_rating": improved_rating,
            "preferred_over_original": preferred,
            "rating_change": improved_rating - original_rating,
        }
    ])

improvement_feedback_df = collect_improvement_feedback(USER_PROMPT, ranking_df, improved_response)
save_feedback(improvement_feedback_df, IMPROVEMENT_FEEDBACK_FILE)

comparison_df = pd.DataFrame([
    {"Metric": "Original winner average rating", "Value": round(float(improvement_feedback_df.iloc[0]["original_average_rating"]), 2)},
    {"Metric": "Improved response rating", "Value": int(improvement_feedback_df.iloc[0]["improved_rating"])},
    {"Metric": "Rating change", "Value": round(float(improvement_feedback_df.iloc[0]["rating_change"]), 2)},
    {"Metric": "Preferred improved response", "Value": "Yes" if int(improvement_feedback_df.iloc[0]["preferred_over_original"]) else "No"},
])

comparison_df

# 6. Image Generation

In [ ]:
from diffusers import StableDiffusionPipeline

def load_diffusion_pipeline(model_id=DIFFUSION_MODEL_ID):
    if DEVICE in {"cuda", "mps"}:
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        ).to(DEVICE)
    else:
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float32,
            use_safetensors=True,
        ).to(DEVICE)

    pipe.enable_attention_slicing()
    if hasattr(pipe, "safety_checker"):
        pipe.safety_checker = None
    return pipe

diffusion_pipe = load_diffusion_pipeline()
print("Stable Diffusion loaded.")

In [ ]:
def generate_image(prompt, output_file=IMAGE_FILE):
    generator_device = "cuda" if DEVICE == "cuda" else "cpu"
    generator = torch.Generator(device=generator_device).manual_seed(SEED)
    steps = 30 if DEVICE == "cuda" else 20 if DEVICE == "mps" else 15

    image = diffusion_pipe(
        prompt,
        num_inference_steps=steps,
        guidance_scale=7.5,
        height=512,
        width=512,
        generator=generator,
    ).images[0]
    image.save(output_file)
    return image

generated_image = generate_image(USER_PROMPT)

plt.figure(figsize=(6, 6))
plt.imshow(generated_image)
plt.axis("off")
plt.title(USER_PROMPT)
plt.show()

print(f"Saved image: {IMAGE_FILE}")

# Final Summary

In [ ]:
summary = {
    "prompt": USER_PROMPT,
    "text_model": TEXT_MODEL_ID,
    "diffusion_model": DIFFUSION_MODEL_ID,
    "feedback_file": str(FEEDBACK_FILE),
    "improvement_feedback_file": str(IMPROVEMENT_FEEDBACK_FILE),
    "image_file": str(IMAGE_FILE),
    "top_response": ranking_df.iloc[0]["response_text"],
    "improved_response": improved_response,
    "improved_response_rating": int(improvement_feedback_df.iloc[0]["improved_rating"]),
    "human_preferred_improved_response": bool(improvement_feedback_df.iloc[0]["preferred_over_original"]),
}

summary_file = OUTPUT_DIR / "presentation_summary.json"
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

display(Markdown("### Final Improved Text"))
display(Markdown(improved_response))
print(f"Saved summary: {summary_file}")